# Analysis Main Notebook

이 노트북은 `Analysis` 안의 분석 도구를 한 곳에서 호출합니다. SSH 환경에서도 결과를 볼 수 있도록 그래프는 기본적으로 `__RESULTS__/_analysis` 아래 PNG/JSON/CSV로 저장합니다.

## 0. 공통 설정

In [ ]:
from pathlib import Path
import json
import pandas as pd

from Analysis.analyzer import Analyzer
from Analysis.distance_metrics import nearestStats
from Analysis.result_io import finalPoints, listBands, loadAlgoRuns
from Analysis.statistics import calcStats, printStats, reportCluster
from Analysis.trends import (
    plotConverge,
    saveReport,
    coverSummary,
)

RESULTS_ROOT = "__RESULTS__"
ALGORITHM = "ga"          # ga, pso, drl, greedy
MAP_NAME = "gangjin.down" # gangjin.down, seocho.up, ...
SEED_BAND = "60-80"      # None이면 전체 band, 단일 run 분석에는 하나의 band 사용
OUTPUT_DIR = "__RESULTS__/_analysis"
GRID_M = 5.0
COVERAGE = 45
TARGET_VALUES = (2, 3)

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
RUN_DIR = Path(RESULTS_ROOT) / ALGORITHM / MAP_NAME / SEED_BAND
RUN_DIR

## 1. 결과 파일과 seed band 확인

In [ ]:
bands = listBands(results_root=RESULTS_ROOT, algorithm=ALGORITHM, map_name=MAP_NAME)
records = loadAlgoRuns(
    results_root=RESULTS_ROOT,
    algorithm=ALGORITHM,
    map_name=MAP_NAME,
    seed_band=SEED_BAND,
)
print("seed bands:", bands)
print("run count in selected band:", len(records))
records[0][0]

## 2. 단일 run 세대별 추이

`Analyzer`는 하나의 JSON run에 대해 센서 수, 커버리지, fitness 추이를 확인합니다.

In [ ]:
analyzer = Analyzer(result_root_path=str(RUN_DIR), file_path=0)
single_run_dir = Path(OUTPUT_DIR) / "single_run"
single_run_dir.mkdir(parents=True, exist_ok=True)

analyzer.plot_evolution_trend(
    xtick_step=50,
    include_corners=True,
    figsize=(10, 3),
    dpi=150,
    show=False,
    save_path=single_run_dir / f"{ALGORITHM}_{MAP_NAME.replace('.', '_')}_{SEED_BAND}_sensor_count.png",
    close=True,
)
analyzer.plot_coverage_trend(
    xtick_step=50,
    figsize=(10, 3),
    dpi=150,
    show=False,
    save_path=single_run_dir / f"{ALGORITHM}_{MAP_NAME.replace('.', '_')}_{SEED_BAND}_coverage.png",
    close=True,
)
analyzer.plot_fitness_trend(
    xtick_step=50,
    figsize=(10, 3),
    dpi=150,
    show=False,
    save_path=single_run_dir / f"{ALGORITHM}_{MAP_NAME.replace('.', '_')}_{SEED_BAND}_fitness.png",
    close=True,
)
print("saved single-run plots to", single_run_dir)

## 3. 선택 band의 기본 통계

In [ ]:
printStats(str(RUN_DIR))
stats_tuple = calcStats(str(RUN_DIR))
stats_tuple

## 4. 센서 간 거리 분석

In [ ]:
reportCluster(str(RUN_DIR), MAP_NAME, grid_m=GRID_M)

distance_rows = []
for path, run in records:
    d = nearestStats(finalPoints(run), grid_m=GRID_M)
    d["run_name"] = run.get("run_name", path.stem)
    distance_rows.append(d)
distance_df = pd.DataFrame(distance_rows).set_index("run_name")
distance_df.round(3).head()

## 5. 초기 seed 센서수별 수렴 분석

`metric='best'`는 세대별 best solution 센서수를 기준으로 수렴을 봅니다. `metric='avg'`로 바꾸면 로그의 평균 센서 수 기준으로 분석합니다.

In [ ]:
safe_map = MAP_NAME.replace("/", "_").replace(".", "_")
convergence = plotConverge(
    results_root=RESULTS_ROOT,
    algorithm=ALGORITHM,
    map_name=MAP_NAME,
    seed_bands=bands,
    include_corners=True,
    metric="best",
    threshold=0.5,
    dpi=300,
    show=False,
    save_path=Path(OUTPUT_DIR) / f"sensor_convergence_{ALGORITHM}_{safe_map}.png",
)
print("convergence generation:", convergence["convergence"]["convergence_gen"])
convergence["convergence"]

## 6. 전체 커버리지와 중첩 커버리지 분석

In [ ]:
report = saveReport(
    results_root=RESULTS_ROOT,
    algorithm=ALGORITHM,
    map_name=MAP_NAME,
    output_dir=OUTPUT_DIR,
    seed_bands=bands,
    include_corners=True,
    metric="best",
    threshold=0.5,
    target_values=TARGET_VALUES,
    dpi=300,
    show=False,
)
print(json.dumps({k: report[k] for k in ["convergence_plot", "coverage_overlap_plot", "coverage_overlap_summary"]}, indent=2))

## 7. 논문 표용 요약 CSV

In [ ]:
summary = coverSummary(
    results_root=RESULTS_ROOT,
    algorithm=ALGORITHM,
    map_name=MAP_NAME,
    seed_bands=bands,
    target_values=TARGET_VALUES,
)
summary_df = pd.DataFrame.from_dict(summary, orient="index")
csv_path = Path(OUTPUT_DIR) / f"coverage_overlap_{ALGORITHM}_{safe_map}.csv"
summary_df.to_csv(csv_path, encoding="utf-8-sig")
print("saved", csv_path)
summary_df[[
    "runs",
    "n_sensors_mean",
    "coverage_percent_mean",
    "overlap_percent_of_target_mean",
    "overlap_percent_of_covered_mean",
    "redundant_hit_percent_mean",
    "logged_final_coverage_mean",
]].round(3)